# Constructive Solid Geometry (CSG) with topologic_fast

This notebook demonstrates Constructive Solid Geometry (CSG) operations using `topologic_fast`.

CSG is a technique used in solid modeling where complex shapes are created by combining simpler primitive shapes using Boolean operations:
- **Union**: Combines two shapes into one
- **Difference**: Subtracts one shape from another
- **Intersection**: Keeps only the overlapping region

Reference: https://en.wikipedia.org/wiki/Constructive_solid_geometry

**Note**: This notebook is adapted from the topologicpy CSG tutorial. The original uses a CSG tree graph data structure,
which is not yet available in topologic_fast. Instead, we demonstrate the core Boolean operations directly.

## Import Libraries

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

print(f"topologic_fast version: {tf.__version__}")

## Visualization Helper Functions

In [ ]:
def get_mesh_data(cell, color='lightblue', opacity=0.7):
    """Convert a Cell to plotly Mesh3d data."""
    mesh = tf.Mesh.ByCell(cell)
    obj_content = mesh.ToOBJ()
    
    vertices = []
    faces = []
    
    for line in obj_content.strip().split('\n'):
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0] == 'v':
            vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
        elif parts[0] == 'f':
            # OBJ faces are 1-indexed
            face_indices = [int(p.split('/')[0]) - 1 for p in parts[1:]]
            if len(face_indices) >= 3:
                faces.append(face_indices[:3])
    
    if not vertices or not faces:
        return None
    
    vertices = np.array(vertices)
    faces = np.array(faces)
    
    return go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=opacity,
        flatshading=True
    )


def get_wireframe(cell, color='black', width=1):
    """Get wireframe edges for a cell."""
    edges = cell.Edges()
    traces = []
    
    for edge in edges:
        start = edge.StartVertex()
        end = edge.EndVertex()
        traces.append(go.Scatter3d(
            x=[start.X(), end.X()],
            y=[start.Y(), end.Y()],
            z=[start.Z(), end.Z()],
            mode='lines',
            line=dict(color=color, width=width),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    return traces


def show_cell(cell, title="Cell", color='lightblue'):
    """Display a single cell."""
    fig = go.Figure()
    
    mesh_data = get_mesh_data(cell, color=color)
    if mesh_data:
        fig.add_trace(mesh_data)
    
    for trace in get_wireframe(cell):
        fig.add_trace(trace)
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        width=700,
        height=500
    )
    
    return fig

## Create Primitive Shapes

We'll create the primitives for our CSG operations:
- Two cylinders that will be unioned to form a cross shape
- A cube from which we'll subtract the cross

In [ ]:
# Create a vertical cylinder (along Z axis)
# Parameters: x, y, z (origin), radius, height, sides
cylinder_1 = tf.Cell.Cylinder(0, 0, -1, 0.4, 2, 32)
print(f"Cylinder 1 (vertical): Volume = {cylinder_1.Volume():.3f}")

# Create a horizontal cylinder (along X axis)
# We create it vertical first then rotate it
cylinder_2 = tf.Cell.Cylinder(0, 0, 0, 0.4, 2, 32)
# Note: topologic_fast currently requires manual rotation
# For now, we'll create it differently or use a box approximation
# Since direct rotation may not be available, we'll use Box for the cross arms
horizontal_arm = tf.Cell.Box(-1, -0.4, -0.4, 2, 0.8, 0.8)
print(f"Horizontal arm (box): Volume = {horizontal_arm.Volume():.3f}")

# Create a cube
cube = tf.Cell.Box(-0.5, -0.5, -0.5, 1, 1, 1)
print(f"Cube: Volume = {cube.Volume():.3f}")

## Visualize the Primitives

In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=['Cylinder (Vertical)', 'Box (Horizontal Arm)', 'Cube']
)

# Cylinder 1
mesh1 = get_mesh_data(cylinder_1, color='cyan', opacity=0.7)
if mesh1:
    mesh1.update(scene='scene1')
    fig.add_trace(mesh1, row=1, col=1)

# Horizontal arm
mesh2 = get_mesh_data(horizontal_arm, color='magenta', opacity=0.7)
if mesh2:
    mesh2.update(scene='scene2')
    fig.add_trace(mesh2, row=1, col=2)

# Cube
mesh3 = get_mesh_data(cube, color='gold', opacity=0.7)
if mesh3:
    mesh3.update(scene='scene3')
    fig.add_trace(mesh3, row=1, col=3)

fig.update_layout(
    title='CSG Primitive Shapes',
    width=1200,
    height=400
)

# Update all scenes
for scene in ['scene1', 'scene2', 'scene3']:
    fig.update_layout(**{scene: dict(aspectmode='data')})

fig.show()

## Boolean Union

Union combines two shapes into a single shape. The result contains all points that are in either shape.

In [ ]:
# Union the vertical cylinder with the horizontal arm to create a cross
cross = tf.Topology.Union(cylinder_1, horizontal_arm)

print(f"Cross (Union of cylinder and box):")
print(f"  Volume: {cross.Volume():.3f}")
print(f"  Number of faces: {len(cross.Faces())}")

# Visualize
fig = show_cell(cross, title="Union: Cylinder + Horizontal Arm = Cross", color='lightgreen')
fig.show()

## Boolean Difference

Difference subtracts one shape from another. The result contains points that are in the first shape but not in the second.

In [ ]:
# Subtract the cross from the cube
result = tf.Topology.Difference(cube, cross)

print(f"Final result (Cube - Cross):")
print(f"  Volume: {result.Volume():.3f}")
print(f"  Number of faces: {len(result.Faces())}")

# Visualize
fig = show_cell(result, title="Difference: Cube - Cross", color='coral')
fig.show()

## Boolean Intersection

Intersection keeps only the overlapping region of two shapes.

In [ ]:
# Create two overlapping boxes for intersection demo
box_a = tf.Cell.Box(0, 0, 0, 2, 2, 2)
box_b = tf.Cell.Box(1, 1, 0, 2, 2, 2)

intersection = tf.Topology.Intersection(box_a, box_b)

print(f"Box A: Volume = {box_a.Volume():.3f}")
print(f"Box B: Volume = {box_b.Volume():.3f}")
print(f"Intersection: Volume = {intersection.Volume():.3f}")

# The intersection of two 2x2x2 boxes offset by 1 in x and y should be a 1x1x2 box
# Volume = 1 * 1 * 2 = 2

In [ ]:
# Visualize the intersection operation
fig = go.Figure()

# Box A (transparent)
mesh_a = get_mesh_data(box_a, color='blue', opacity=0.3)
if mesh_a:
    mesh_a.name = 'Box A'
    fig.add_trace(mesh_a)

# Box B (transparent)
mesh_b = get_mesh_data(box_b, color='red', opacity=0.3)
if mesh_b:
    mesh_b.name = 'Box B'
    fig.add_trace(mesh_b)

# Intersection (solid)
mesh_int = get_mesh_data(intersection, color='green', opacity=0.9)
if mesh_int:
    mesh_int.name = 'Intersection'
    fig.add_trace(mesh_int)

fig.update_layout(
    title='Boolean Intersection: Box A (blue) AND Box B (red) = Green',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=800,
    height=600
)

fig.show()

## Complex CSG Example: Building with Windows

Let's create a more complex example: a building with window cutouts.

In [ ]:
# Create main building
building = tf.Cell.Box(0, 0, 0, 10, 8, 6)
print(f"Building volume: {building.Volume():.1f}")

# Create window cutouts (slightly larger than wall thickness to ensure they cut through)
windows = []

# Front windows (y = 0 face)
for x in [1.5, 4, 6.5]:
    for z in [1.5, 3.5]:
        window = tf.Cell.Box(x, -0.5, z, 2, 1, 1.5)
        windows.append(window)

# Back windows (y = 8 face)
for x in [1.5, 4, 6.5]:
    for z in [1.5, 3.5]:
        window = tf.Cell.Box(x, 7.5, z, 2, 1, 1.5)
        windows.append(window)

print(f"Created {len(windows)} window cutouts")

# Subtract all windows from building
result = building
for i, window in enumerate(windows):
    result = tf.Topology.Difference(result, window)
    if (i + 1) % 4 == 0:
        print(f"  Subtracted {i + 1} windows...")

print(f"\nFinal building volume: {result.Volume():.1f}")
print(f"Volume removed: {building.Volume() - result.Volume():.1f}")

In [ ]:
# Visualize the building with windows
fig = show_cell(result, title="Building with Window Cutouts (CSG Difference)", color='sandybrown')
fig.update_layout(width=900, height=600)
fig.show()

## Chained Boolean Operations

Demonstrate chaining multiple operations together.

In [ ]:
# Create base shapes
base = tf.Cell.Box(0, 0, 0, 4, 4, 1)  # Base plate
pillar1 = tf.Cell.Box(0, 0, 1, 1, 1, 3)  # Corner pillars
pillar2 = tf.Cell.Box(3, 0, 1, 1, 1, 3)
pillar3 = tf.Cell.Box(0, 3, 1, 1, 1, 3)
pillar4 = tf.Cell.Box(3, 3, 1, 1, 1, 3)
roof = tf.Cell.Box(0, 0, 4, 4, 4, 0.5)  # Roof

# Chain unions to build structure
structure = base
for pillar in [pillar1, pillar2, pillar3, pillar4]:
    structure = tf.Topology.Union(structure, pillar)
structure = tf.Topology.Union(structure, roof)

# Subtract a cylindrical hole through the roof
hole = tf.Cell.Cylinder(2, 2, 3.5, 0.5, 2, 32)
final_structure = tf.Topology.Difference(structure, hole)

print(f"Structure volume: {structure.Volume():.1f}")
print(f"Hole volume: {hole.Volume():.3f}")
print(f"Final volume: {final_structure.Volume():.1f}")

In [ ]:
# Visualize the chained operations result
fig = show_cell(final_structure, title="Chained CSG: Base + Pillars + Roof - Hole", color='steelblue')
fig.update_layout(width=800, height=600)
fig.show()

## Note on CSG Tree Graphs

The original topologicpy tutorial uses a CSG tree graph data structure (`CSG.Init()`, `CSG.AddTopologyVertex()`, 
`CSG.AddOperationVertex()`, etc.) which provides a declarative way to build complex CSG operations.

**This feature is not yet available in topologic_fast.**

The core Boolean operations (`Union`, `Difference`, `Intersection`) demonstrated above provide the same
functionality, but require imperative chaining rather than building a tree structure.

If you need CSG tree graph functionality, consider:
1. Using the original topologicpy library
2. Building a simple wrapper that provides tree-like semantics using these primitives

## Summary

This notebook demonstrated:

1. **Boolean Union** (`tf.Topology.Union`) - Combine shapes
2. **Boolean Difference** (`tf.Topology.Difference`) - Subtract shapes
3. **Boolean Intersection** (`tf.Topology.Intersection`) - Find overlap
4. **Chained Operations** - Building complex geometry step by step

### Key API Methods:
- `tf.Cell.Box(x, y, z, width, length, height)` - Create a box
- `tf.Cell.Cylinder(x, y, z, radius, height, sides)` - Create a cylinder
- `tf.Topology.Union(a, b)` - Boolean union
- `tf.Topology.Difference(a, b)` - Boolean difference
- `tf.Topology.Intersection(a, b)` - Boolean intersection